# Forecasting Maize Prices with Prophet

Random Forest failed to beat a simple 3-month rolling average (MAE 700 vs 196).
The root cause: Random Forest can't extrapolate beyond its training range, and prices
corrected sharply in 2024 after a 2022–2023 spike.

Prophet is a statistical forecasting tool built by Meta, designed specifically for
time series with trend, seasonality, and structural breaks. We use the same
train/test split: train on July 2017–December 2023, test on January 2024–April 2025.

The benchmark to beat: MAE 196 UGX/kg (naive rolling mean baseline).


In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "prophet"], check=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 3.8 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 6.4 MB/s eta 0:00:00 MB/s eta 0:00:01



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


CompletedProcess(args=['/Users/pascal/projects/east-africa-food-prices/.venv/bin/python', '-m', 'pip', 'install', 'prophet'], returncode=0)

## Load and Prepare Data

Prophet expects a DataFrame with exactly two columns:
- `ds` — the date column
- `y` — the value to forecast

We load the clean retail maize prices and CPI, merge them, and rename columns to match Prophet's format.


In [ ]:
import pandas as pd
from prophet import Prophet
from huggingface_hub import hf_hub_download

# Download datasets from HuggingFace
prices_path = hf_hub_download(
    repo_id="byabasaija/uganda-maize-prices",
    filename="uganda_maize_prices_owino.csv",
    repo_type="dataset",
)
cpi_path = hf_hub_download(
    repo_id="byabasaija/uganda-food-cpi",
    filename="food_cpi_monthly.csv",
    repo_type="dataset",
)

maize_df = pd.read_csv(prices_path)
cpi_df = pd.read_csv(cpi_path)

# Prophet requires ds and y column names
maize_df["ds"] = pd.to_datetime(maize_df[["year", "month"]].assign(day=15))
df = maize_df.merge(cpi_df, on=["year", "month"], how="inner")
df = df.rename(columns={"price": "y"})[["ds", "y", "food_cpi"]]

print(f"Rows: {len(df)}")
print(f"Date range: {df['ds'].min().date()} to {df['ds'].max().date()}")
df.head()

In [3]:
train = df[df["ds"] < "2024-01-01"]
test = df[df["ds"] >= "2024-01-01"]

print(f"Train: {len(train)} rows")
print(f"Test:  {len(test)} rows")


Train: 71 rows
Test:  16 rows


## Train Prophet

We add `food_cpi` as an extra regressor — an additional signal beyond the time series itself.
Prophet will learn the relationship between CPI and price alongside the trend and seasonality.


In [4]:
model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model.add_regressor("food_cpi")
model.fit(train)
print("Model trained.")


06:53:54 - cmdstanpy - INFO - Chain [1] start processing
06:53:57 - cmdstanpy - INFO - Chain [1] done processing


Model trained.


## Evaluate on Test Set

We generate predictions for the test period and compare against actual prices.


In [5]:
from sklearn.metrics import mean_absolute_error, r2_score

future = test[["ds", "food_cpi"]].copy()
forecast = model.predict(future)

y_pred = forecast["yhat"].values
y_test = test["y"].values

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:,.0f} UGX/kg")
print(f"R²  : {r2:.4f}")
print(f"\nBenchmark to beat — Naive rolling mean: MAE 196 UGX/kg")


MAE : 1,169 UGX/kg
R²  : -24.1831

Benchmark to beat — Naive rolling mean: MAE 196 UGX/kg


In [6]:
print("Actual vs Predicted:")
for actual, pred in zip(y_test, y_pred):
    print(f"  Actual: {actual:,.0f}  Predicted: {pred:,.0f}  Diff: {actual-pred:+,.0f}")


Actual vs Predicted:
  Actual: 2,225  Predicted: 3,394  Diff: -1,169
  Actual: 1,938  Predicted: 3,214  Diff: -1,276
  Actual: 2,375  Predicted: 3,058  Diff: -683
  Actual: 2,062  Predicted: 3,172  Diff: -1,110
  Actual: 2,000  Predicted: 3,545  Diff: -1,545
  Actual: 2,000  Predicted: 2,929  Diff: -929
  Actual: 1,700  Predicted: 3,688  Diff: -1,988
  Actual: 1,872  Predicted: 2,947  Diff: -1,075
  Actual: 1,912  Predicted: 3,167  Diff: -1,255
  Actual: 1,838  Predicted: 2,992  Diff: -1,155
  Actual: 2,000  Predicted: 3,450  Diff: -1,450
  Actual: 2,000  Predicted: 3,579  Diff: -1,579
  Actual: 2,000  Predicted: 2,870  Diff: -870
  Actual: 2,500  Predicted: 3,451  Diff: -951
  Actual: 2,282  Predicted: 3,250  Diff: -967
  Actual: 2,615  Predicted: 3,316  Diff: -701


In [7]:
model_flat = Prophet(
    growth="flat",
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)
model_flat.add_regressor("food_cpi")
model_flat.fit(train)

forecast_flat = model_flat.predict(future)
y_pred_flat = forecast_flat["yhat"].values

mae_flat = mean_absolute_error(y_test, y_pred_flat)
r2_flat = r2_score(y_test, y_pred_flat)

print(f"Prophet (flat trend) — MAE: {mae_flat:,.0f}  R²: {r2_flat:.4f}")
print(f"Prophet (with trend) — MAE: {mae:,.0f}  R²: {r2:.4f}")
print(f"Naive baseline       — MAE: 196")


07:02:30 - cmdstanpy - INFO - Chain [1] start processing
07:02:30 - cmdstanpy - INFO - Chain [1] done processing


Prophet (flat trend) — MAE: 1,187  R²: -24.8943
Prophet (with trend) — MAE: 1,169  R²: -24.1831
Naive baseline       — MAE: 196


In [ ]:
maize_full = pd.read_csv(prices_path)
maize_full["ds"] = pd.to_datetime(maize_full[["year", "month"]].assign(day=15))
maize_full = maize_full.rename(columns={"price": "y"})[["ds", "y", "year", "month"]]

train_full = maize_full[maize_full["ds"] < "2024-01-01"]
test_full = maize_full[maize_full["ds"] >= "2024-01-01"]

model_full = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
model_full.fit(train_full[["ds", "y"]])

forecast_full = model_full.predict(test_full[["ds"]])
mae_full = mean_absolute_error(test_full["y"].values, forecast_full["yhat"].values)
r2_full = r2_score(test_full["y"].values, forecast_full["yhat"].values)

print(f"Prophet (full data, no CPI) — MAE: {mae_full:,.0f}  R²: {r2_full:.4f}")
print(f"Naive baseline             — MAE: 196")

## Test Original Model on Full 2020–2025 Period

The original model was only tested on 2020–2022. Let's see how it holds up on the full
unseen period including the 2022–2023 spike and the 2024 correction.


In [ ]:
import joblib
import sys
sys.path.insert(0, "../food_prices_api")
from features import compute_features
from huggingface_hub import hf_hub_download as hf_download

original_model_path = hf_download(
    repo_id="byabasaija/uganda-food-prices-model",
    filename="maize_price_model.pkl",
)
original_model = joblib.load(original_model_path)

prices_series = maize_full["y"].tolist()
feature_rows = []
targets = []
dates = []

FEATURE_COLS = ["price_lag_1", "price_lag_2", "price_lag_3",
                "rolling_mean_3", "rolling_mean_6", "month", "year"]

for i in range(6, len(maize_full)):
    past_prices = prices_series[i - 6 : i]
    row = maize_full.iloc[i]
    feats = compute_features(past_prices, int(row["month"]), int(row["year"]))
    feature_rows.append(feats)
    targets.append(prices_series[i])
    dates.append(row["ds"])

X_all = pd.DataFrame(feature_rows)[FEATURE_COLS]
y_all = pd.Series(targets)
dates = pd.Series(dates)

test_mask = dates >= "2020-01-01"
y_pred_orig = original_model.predict(X_all[test_mask])
mae_orig = mean_absolute_error(y_all[test_mask], y_pred_orig)
r2_orig = r2_score(y_all[test_mask], y_pred_orig)

print(f"Original RF — test on 2020–2025 — MAE: {mae_orig:,.0f}  R²: {r2_orig:.4f}")
print(f"Original RF — test on 2020–2022 — MAE: 193")

In [10]:
test_2024_mask = dates >= "2024-01-01"
y_pred_2024 = original_model.predict(X_all[test_2024_mask])
mae_2024 = mean_absolute_error(y_all[test_2024_mask], y_pred_2024)
r2_2024 = r2_score(y_all[test_2024_mask], y_pred_2024)

print(f"Original RF — test on 2024–2025 only — MAE: {mae_2024:,.0f}  R²: {r2_2024:.4f}")
print(f"RF v2 with CPI — test on 2024–2025  — MAE: 658")
print(f"Naive baseline                       — MAE: 196")


Original RF — test on 2024–2025 only — MAE: 837  R²: -11.4921
RF v2 with CPI — test on 2024–2025  — MAE: 658
Naive baseline                       — MAE: 196
